# Data Analysis

Answer the selected question, “Which delivery time slots cause the most late orders?”, using approval time as the time-slot anchor. Delivery outcomes count distinct orders.

**Contents**

1. [Project setup and input](#analysis-setup) — Load the cleaned item table.
2. [Order-level delivery outcomes](#analysis-orders) — Classify orders, assign approval slots and calculate lateness.
3. [Late delivery by approval slot](#analysis-slot-summary) — Compare order counts, late rates and delay duration.
4. [Delivery patterns in figures](#analysis-figures) — Inspect four slots, hourly and weekday rates, and delay duration.
5. [Statistical comparison of slots](#analysis-statistics) — Test equal late rates and assess the size of the difference.
6. [Temporal and cohort stability](#analysis-stability) — Compare months, matched years and cleaning cohorts.
7. [Delivery stages and promised dates](#analysis-stages) — Compare elapsed time, eligible-day time and delivery promises.
8. [Route composition and delivery](#analysis-routes) — Compare route types, approval slots and carrier transit.
9. [Focus cohort: 12:00–18:00](#analysis-focus) — Prepare the order cohort for route, time and destination comparisons.
10. [Destination states in the focus cohort](#analysis-destinations) — Locate late orders across all routes and within inter-state deliveries.
11. [Seller and customer state pairs](#analysis-state-pairs) — Identify late order concentrations within inter-state deliveries.
12. [Time patterns in the focus cohort](#analysis-focus-time) — Compare months, approval hours and weekdays for inter-state orders and the leading state pair.
13. [Approval slots within the leading state pair](#analysis-pair-slots) — Compare all four slots for the selected route, including matched purchase months.
14. [Route stage comparison](#analysis-route-stages) — Compare delivery stages and promised windows within purchase periods.
15. [Evidence across slots, routes and purchase periods](#analysis-synthesis) — Connect matched route outcomes, delivery stages and promises.
16. [Orders approved from 00:00–06:00](#analysis-overnight) — Inspect overnight route types, destinations and state-pair coverage.
17. [Overnight and afternoon approval cohorts](#analysis-two-slot-comparison) — Compare selected groups, late rates and Wilson intervals.
18. [Purchase-month comparisons across approval slots](#analysis-slot-months) — Inspect leader groups and compare common-month weights.
19. [Delivery stages and promises across the two slots](#analysis-slot-stage-context) — Reuse SP → RJ comparisons and examine SP → SP within common purchase months.
20. [Answer to the delivery question](#analysis-answer) — Connect the count and rate priorities with route, month and delivery-stage evidence.

The slot test uses the summary in section 3, the cleaning comparison also reads the full merged table, and the route comparison uses the stage durations from section 7.

[Cleaning User Guide](../docs/Cleaning/User_Guide.md) · [Cleaning Technical Reference](../docs/Cleaning/Technical_Reference.md)


<a id="analysis-setup" name="analysis-setup"></a>

## 1. Project setup and input

Locate the project and load the cleaned order-item table.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    # Resolve imports from either the repository root or notebooks directory.
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import matplotlib.pyplot as plt

from src.analysis.presentation import format_analysis_table
from src.analysis.orders import build_order_analysis_view
from src.analysis.summary import summarize_delivery_by_slot
from src.analysis.statistics import test_slot_association
from src.analysis.slot_comparisons import (
    COMPARISON_SLOTS, compare_approval_slot_groups, summarize_slot_group_months,
)

from src.analysis.plots import (
    build_temporal_profiles, plot_slot_counts, plot_rate_profile,
    plot_hour_weekday_heatmap, plot_delay_distribution,
)
from src.analysis.route_slots import build_state_pair_orders, summarize_state_pair_slots
from src.analysis.route_stages import (
    compare_route_delivery_stages, summarize_afternoon_route_evidence, summarize_route_slot_evidence,
    summarize_stage_slots,
)
from src.data_loader import load_data
from src.paths import PROCESSED_DIR

from src.analysis.investigation import (
    summarize_time_periods, compare_cleaning_cohorts, investigate_delivery_stages,
    classify_order_routes, summarize_routes, summarize_focus_routes, build_focus_cohort, summarize_destination_states,
    summarize_state_pairs, summarize_focus_time, summarize_focus_comparisons,
)
from src.analysis.investigation_plots import (
    plot_monthly_slot_rates, plot_matched_years, plot_stage_comparison, plot_promised_windows,
    plot_slot_group_months, plot_slot_group_comparison, plot_focus_routes, plot_route_slot_rates, plot_route_transit_time, plot_destination_states, plot_state_pairs,
    plot_focus_months, plot_focus_approval_time,
    plot_state_pair_slots, plot_state_pair_months,
    plot_route_period_medians,
    plot_route_period_slot_rates,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed input: {PROCESSED_DIR}")


### Load the cleaned item table


In [ ]:
cleaned_items = load_data("cleaned_order_items", stage="processed")


<a id="analysis-orders" name="analysis-orders"></a>

## 2. Order-level delivery outcomes

Keep each order once in the analysis view. Orders with missing or conflicting order-level values retain all their item rows in the review view. One reviewed order may have several reasons. Approval time determines one of four six-hour slots. A delivery is late only when its calendar date is after the estimated date.


In [ ]:
orders, order_diagnostics = build_order_analysis_view(cleaned_items)
display(order_diagnostics["cohort_summary"])

if not order_diagnostics["review_reasons"].empty:
    print("Orders requiring review by field and issue")
    display(order_diagnostics["review_reasons"].groupby(["field", "issue"]).size().rename("orders").reset_index())
    print("First 20 review reasons; complete records remain in order_diagnostics['review_reasons'].")
    display(order_diagnostics["review_reasons"].head(20))
    print("Original item rows remain in order_diagnostics['review_items'].")

print("First five analyzable orders")
display(orders[[
    "order_id", "order_approved_at", "approval_slot",
    "order_estimated_delivery_date", "order_delivered_customer_date",
    "is_late", "delay_days",
]].head())


<a id="analysis-slot-summary" name="analysis-slot-summary"></a>

## 3. Late delivery by approval slot

Compare the four approval-time slots at order grain. Late rate divides late orders by all analyzable orders in the slot. Mean and median delay use late orders only.


In [ ]:
slot_summary = summarize_delivery_by_slot(orders)
display(format_analysis_table(slot_summary))

if orders.empty:
    print("No analyzable orders are available for slot comparisons.")
elif not orders["is_late"].any():
    print("No late orders are available for a late-delivery ranking.")
else:
    for label, measure in [
        ("Most late orders", "late_orders"),
        ("Highest late rate", "late_rate_pct"),
        ("Longest mean delay among late orders", "mean_delay_days"),
    ]:
        maximum = slot_summary[measure].max()
        leaders = slot_summary.loc[slot_summary[measure].eq(maximum), "approval_slot"]
        print(f"{label}: {', '.join(leaders)}")


For the current cleaned dataset, 12:00–18:00 has the most late orders (2,085) and the largest order volume. The highest late rate is 00:00–06:00 (7.31%). Among late orders, 06:00–12:00 has the longest mean delay (11.18 days), while every slot has a median of seven days. These rankings describe different aspects of delivery performance. The 00:00–06:00 cohort is examined separately in [section 16](#analysis-overnight).


<a id="analysis-figures" name="analysis-figures"></a>

## 4. Delivery patterns in figures

Use the same order-level data as the summary table. Each rate uses orders in its own group as the denominator. Hour and weekday profiles include every recorded hour and weekday.


In [ ]:
temporal_profiles = build_temporal_profiles(orders)


### On-time and late deliveries by approval slot

Each stack counts all delivered orders approved in one slot. The labels give the total and the number delivered late.


In [ ]:
slot_counts_figure = plot_slot_counts(slot_summary)
display(slot_counts_figure)
plt.close(slot_counts_figure)


The 12:00–18:00 slot has both the largest order volume and the most late orders. Its late rate remains close to the overall rate, as shown in the summary table.


### Late delivery rate by approval hour

Each rate uses the orders approved in that hour of the recorded clock. N gives the number of orders.


In [ ]:
hourly_rate_figure = plot_rate_profile(temporal_profiles["hours"], "approval_hour")
display(hourly_rate_figure)
plt.close(hourly_rate_figure)


Among individual hours, 03:00–03:59 has the highest observed late rate (7.85%, 422 of 5,378 orders). The busiest approval hour is 02:00–02:59, with 6,855 orders and a 7.66% late rate.


### Late delivery rate by approval weekday

Each rate uses all orders approved on that weekday, across all hours. N gives the number of orders.


In [ ]:
weekday_rate_figure = plot_rate_profile(temporal_profiles["weekdays"], "approval_weekday")
display(weekday_rate_figure)
plt.close(weekday_rate_figure)


Across all approval hours, Friday has the highest observed weekday late rate (7.43%, 1,049 of 14,119 orders).


### Late delivery rates and order counts by approval hour and weekday

Darker green indicates a higher share of late deliveries. Darker blue indicates more orders. Grey marks an hour and weekday with no orders. The table lists the ten highest observed rates with their order counts.


In [ ]:
hour_weekday_figure = plot_hour_weekday_heatmap(temporal_profiles["hour_weekday"])
display(hour_weekday_figure)
plt.close(hour_weekday_figure)
display(
    temporal_profiles["hour_weekday"].sort_values("late_rate_pct", ascending=False)
    .head(10).rename(columns={
        "approval_weekday": "Weekday (Mon=0)",
        "approval_hour": "Approval hour",
        "orders": "Orders",
        "late_orders": "Late orders",
        "late_rate_pct": "Late rate (%)",
    }).style.format({"Late rate (%)": "{:.2f}", "Orders": "{:,.0f}", "Late orders": "{:,.0f}"})
    .hide(axis="index")
)


The highest observed hour-by-weekday rate is Sunday 00:00–00:59, at 12.66% (30 of 237 orders). The paired count panel shows why a high-rate cell should be read alongside its sample size.


### Days past the promised date among late orders

The boxplots show calendar days late by approval slot. One view covers the full distribution. The detailed view ends at the 99th percentile across all late orders. Both use the full samples to calculate the boxes, medians and means.


In [ ]:
delay_distribution_figure = plot_delay_distribution(orders)
display(delay_distribution_figure)
plt.close(delay_distribution_figure)


Every approval slot has a median delay of seven days among late orders. The means differ more, so the upper tail contributes to the severity ranking.


<a id="analysis-statistics" name="analysis-statistics"></a>

## 5. Statistical comparison of slots

The Chi-square test evaluates whether late-delivery rates differ across the four approval slots. The null hypothesis is that all four rates are equal. The alternative is that at least one rate differs. The significance level is 5%.


### Observed delivery outcomes

The contingency table counts on-time and late deliveries in each approval slot. The test derives expected counts from the row and column totals under the assumption of equal late rates.


In [ ]:
slot_test = test_slot_association(slot_summary, alpha=0.05)
display(slot_test["observed"].rename_axis("Approval slot").rename(columns={
    "on_time_orders": "On time", "late_orders": "Late",
}).style.format("{:,.0f}"))
failed_checks = slot_test["checks"].loc[lambda table: ~table["passed"]]
if not failed_checks.empty:
    display(failed_checks[["check"]].rename(columns={"check": "Unmet test condition"}).style.hide(axis="index"))


### Test result and size of the association

The p-value measures how unusual the overall difference would be under equal rates and the test assumptions. Cramer's V describes the strength of association. The percentage-point gap compares the highest and lowest observed rates.


In [ ]:
test_result = slot_test["result"].iloc[0]
test_metrics = slot_test["result"][[
    "orders", "alpha", "chi2", "dof", "p_value",
    "min_expected", "cramers_v", "rate_range_pp",
]].rename(columns={
    "orders": "Analyzable orders",
    "alpha": "Significance threshold",
    "chi2": "Chi-square statistic",
    "dof": "Degrees of freedom",
    "p_value": "P-value",
    "min_expected": "Smallest expected count",
    "cramers_v": "Cramér's V",
    "rate_range_pp": "Highest minus lowest rate (percentage points)",
}).T
test_metrics.columns = ["Value"]
display(test_metrics.style.format("{:,.6g}", na_rep="—"))
print(test_result["decision"])


Late-delivery rates differ across approval slots in the analyzed cohort, with Chi-square = 13.78, 3 degrees of freedom and p = 0.0032. The minimum expected count is 1,233, satisfying the frequency criterion.

The largest observed rate difference is 0.96 percentage points, and Cramer's V is 0.012. This indicates a small association. The large sample provides evidence of a difference even though its absolute size is modest.


### Interpretation limits

The Chi-square p-value applies to the four-slot table as a whole. The slot rankings describe observed rates; the overall test does not establish which individual pairs differ.

Orders sharing an approval date or seller may be exposed to common operating conditions, which can violate the independence assumption. The p-value therefore depends on an assumption that cannot be fully verified from these records.

The findings apply to delivered orders retained after cleaning. Differences in season, geography or promised delivery time may also contribute to the observed association. The results alone do not establish the effect of changing warehouse or carrier schedules.


<a id="analysis-stability" name="analysis-stability"></a>

## 6. Temporal and cohort stability

Compare slot rates across purchase months, matched January-July periods, and before versus after cleaning. Purchase dates select the time cohort; approval times assign the slots.


### Monthly late delivery rates and order counts by approval slot

Orders are grouped by purchase month and colored by approval slot. Rates are shown on the full observed range and a 0–25% scale. The count series gives the denominator for each month and slot. A gap in a rate series indicates zero orders.


In [ ]:
period_profiles = summarize_time_periods(orders)
monthly_counts = period_profiles["monthly"].groupby("month", as_index=False)[["orders", "late_orders"]].sum()
display(format_analysis_table(monthly_counts))


In [ ]:
analysis_monthly_figure = plot_monthly_slot_rates(period_profiles["monthly"])
display(analysis_monthly_figure)
plt.close(analysis_monthly_figure)


September 2016 contains one order and November 2016 has none. August 2018 covers purchases through 29 August and contains 6,261 orders. The matched-year comparison uses January–July in both years.


### Late delivery rates for January–July purchases in 2017 and 2018

Each bar uses orders purchased in January–July of the indicated year. Approval time determines the slot. N gives the number of orders in that group.


In [ ]:
display(format_analysis_table(period_profiles["matched_years"][[
    "year", "approval_slot", "orders", "late_orders", "late_rate_pct",
]]))


In [ ]:
analysis_years_figure = plot_matched_years(period_profiles["matched_years"])
display(analysis_years_figure)
plt.close(analysis_years_figure)


In January-July 2017, 18:00–24:00 has the highest rate (3.88%), closely followed by 00:00–06:00 (3.85%). In the same months of 2018, 00:00–06:00 ranks highest (8.45%). Every slot has a higher observed rate in 2018, so the pooled ranking also reflects the time composition of orders.


### Before and after cleaning

The comparison starts with delivered orders whose approval, customer-delivery and estimated-delivery timestamps are available and consistent within the order. Compare their four-slot rates with the analyzable cleaned cohort. Missing comparison timestamps and conflicting orders are counted separately.


In [ ]:
merged_items = load_data("merged_order_items", stage="interim")
cohort_comparison = compare_cleaning_cohorts(merged_items, orders)
display(format_analysis_table(cohort_comparison["coverage"]))
if cohort_comparison["slots"].empty:
    print("Comparison unavailable. Check the missing or changed orders in the coverage table.")
else:
    display(format_analysis_table(cohort_comparison["slots"][[
        "cohort", "approval_slot", "orders", "late_orders", "late_rate_pct",
    ]]))


For the analyzed Olist extract, 22 of 96,478 delivered orders lack at least one comparison timestamp. The comparison therefore uses 96,456 orders before cleaning and 95,075 after cleaning. The 12:00–18:00 slot has the most late orders and 00:00–06:00 has the highest late rate in both cohorts.


<a id="analysis-stages" name="analysis-stages"></a>

## 7. Delivery stages and promised dates

Measure approval to carrier handoff, then carrier handoff to customer delivery. Recorded elapsed hours preserve the full wait. Eligible-day hours count Monday-Friday except 1 January, 1 May and 25 December, with 24 hours per eligible day and partial endpoint days retained. These hours describe a calendar convention rather than actual staffed hours.


In [ ]:
stage_profiles = investigate_delivery_stages(orders)
display(format_analysis_table(stage_profiles["quality"]))
display(format_analysis_table(stage_profiles["summary"]))
print("Median hours across all approval slots")
display(format_analysis_table(stage_profiles["pooled_medians"]))


### Time from approval to carrier handoff

Bars show the median time to handoff by approval slot for on-time and late deliveries. N counts orders with a valid interval in each group.


In [ ]:
analysis_processing_figure = plot_stage_comparison(stage_profiles["summary"], "Processing")
display(analysis_processing_figure)
plt.close(analysis_processing_figure)


Across all slots, median recorded approval-to-handoff time is 73.60 hours for late orders and 43.01 hours for on-time orders. Under the eligible-day convention, the corresponding medians are 50.15 and 30.10 hours. These retrospective groups differ in the time before handoff.


### Time from carrier handoff to customer delivery

Bars show the median transit time by approval slot and delivery outcome. Elapsed hours cover the full interval. Eligible hours count the part of that interval on eligible days.


In [ ]:
analysis_transit_figure = plot_stage_comparison(stage_profiles["summary"], "Carrier transit")
display(analysis_transit_figure)
plt.close(analysis_transit_figure)


Median recorded carrier-to-customer time is 628.75 hours for late orders and 166.95 hours for on-time orders. Eligible-day medians are 438.05 and 118.22 hours. Late status also depends on the promised date, so these duration differences alone cannot identify the cause of a late delivery.


### Days from approval to the promised delivery date

The interval uses calendar dates. A negative value identifies an order approved after its promised date. The boxplots show the full range by approval slot.


In [ ]:
display(format_analysis_table(stage_profiles["promised_summary"]))
print(f"Orders approved after the estimated delivery date: {len(stage_profiles['negative_windows']):,}")
if not stage_profiles["negative_windows"].empty:
    display(format_analysis_table(stage_profiles["negative_windows"].head(10)))


In [ ]:
analysis_promise_figure = plot_promised_windows(stage_profiles["windows"])
display(analysis_promise_figure)
plt.close(analysis_promise_figure)


Median promised windows are 22, 23, 24 and 24 calendar days for the four slots, respectively. The earliest approval slot has a shorter typical promise window, which should be considered alongside its higher late rate. One negative window is retained in the displayed distribution and review table.


<a id="analysis-routes" name="analysis-routes"></a>

## 8. Route composition and delivery

Classify each analyzed order from all its item-level seller and customer states. Intra-state means every item has matching states; inter-state means every item differs. Mixed orders contain both patterns. Any missing state makes the route unknown. State agreement is a geographic proxy, not a measured distance or actual carrier route.


### Route coverage and approval slots

Count each order once in its route group. The four-slot table retains empty groups with undefined rates. Each late rate divides late orders by all orders in that route and slot.


In [ ]:
order_routes = classify_order_routes(cleaned_items, orders)
route_profiles = summarize_routes(orders, order_routes, stage_profiles["records"])
print("Route totals")
display(format_analysis_table(route_profiles["totals"]))
print("Route groups by approval slot")
display(format_analysis_table(route_profiles["slots"]))


The 60,776 inter-state orders include 4,950 late deliveries (8.14%), compared with 1,558 of 34,059 intra-state orders (4.57%). The observed gap is 3.57 percentage points. The 240 mixed orders include one late delivery; there are no unknown routes in this cohort. The inter-state group accounts for 76.05% of late orders and 63.92% of all orders. These differences describe route composition, without separating it from other factors associated with destinations or sellers.


### Intra-state and inter-state late delivery rates by approval slot

The rate uses all orders in each route and slot. Intra-state orders have matching seller and customer states on every item. Inter-state orders have different states on every item.


In [ ]:
analysis_route_slot_figure = plot_route_slot_rates(route_profiles["slots"])
display(analysis_route_slot_figure)
plt.close(analysis_route_slot_figure)


Within intra-state orders, 00:00–06:00 has the highest observed late rate (5.46%, 393 of 7,198). Within inter-state orders, 00:00–06:00 is also highest (8.40%, 1,062 of 12,649), closely followed by 18:00–24:00 (8.37%, 1,433 of 17,112). The slot ranking is therefore less distinct within inter-state deliveries than the pooled ranking alone suggests.


### Median carrier transit time by route type

The table covers all route types. The figure compares intra-state and inter-state orders using elapsed hours and eligible hours. N counts orders with a valid handoff-to-delivery interval.


In [ ]:
display(format_analysis_table(route_profiles["transit"]))


In [ ]:
analysis_route_transit_figure = plot_route_transit_time(route_profiles["transit"])
display(analysis_route_transit_figure)
plt.close(analysis_route_transit_figure)


Median carrier transit is 87.77 recorded hours for intra-state orders and 221.04 for inter-state orders. Under the eligible-day convention, the medians are 64.55 and 164.25 hours. All 95,075 orders have valid transit intervals. Longer observed transit in the inter-state group is consistent with the geographic proxy, but state equality alone does not measure distance or establish why an individual order was late.


<a id="analysis-focus" name="analysis-focus"></a>

## 9. Focus cohort: 12:00–18:00

Retain all analyzed orders approved in this slot. Assign a customer state only when every item has the same nonblank value, and list unresolved destinations separately. Orders retain the route classifications established above.

The result commentary in sections 9–12 describes the reported Olist extract. Output tables reflect the data loaded during the current run.


In [ ]:
focus_orders, focus_diagnostics = build_focus_cohort(cleaned_items, orders, order_routes)
display(format_analysis_table(focus_diagnostics["summary"]))

if focus_diagnostics["location_review"].empty:
    print("No customer states require review in this cohort.")
else:
    print("Customer states requiring review")
    display(format_analysis_table(focus_diagnostics["location_review"]))

display(format_analysis_table(route_profiles["slots"].loc[
    route_profiles["slots"]["approval_slot"].eq("12:00–18:00")
]))


The cohort contains 30,448 orders, including 2,085 late deliveries (6.85%), with a known customer state for every order. Inter-state deliveries account for 63.81% of orders and 75.92% of late deliveries in this slot. Their late rate is 8.15%, compared with 4.58% for intra-state orders.


<a id="analysis-destinations" name="analysis-destinations"></a>

## 10. Destination states in the focus cohort

Group the 12:00–18:00 orders by customer state, first across all route types and then within inter-state deliveries. Late rate uses all orders to the state; share of scope late orders uses all late orders in the respective scope, including destinations requiring review.


### All route types

Rank states by late order count across the full focus cohort. Unresolved destinations remain in the table as Requires review.


In [ ]:
destination_profiles = summarize_destination_states(focus_orders)
display(format_analysis_table(destination_profiles["all_routes"]))


SP receives the most late orders across all route types: 580 of 12,999 orders (4.46%). RJ receives 502 late orders out of 4,061 (12.36%). SP has the larger late count and a larger order volume, while late delivery is more prevalent among orders to RJ.


### Inter-state orders

Restrict the comparison to inter-state deliveries in the same approval slot. Each state's late share now uses the total late inter-state orders as its denominator.


In [ ]:
display(format_analysis_table(destination_profiles["inter_state"]))


The inter-state subset contains 19,430 orders and 1,583 late deliveries. RJ receives 489 late orders out of 3,736 (13.09%), representing 19.23% of inter-state orders and 30.89% of late inter-state orders in the slot. Inter-state deliveries account for 120 of the 580 late orders to SP (20.69%), compared with 489 of the 502 to RJ (97.41%). This route composition explains the change in the leading destination from SP across all routes to RJ within inter-state deliveries.


### Leading destinations for late inter-state orders

Compare the five states with the most late inter-state orders. Both panels use the same state ranking; N gives all inter-state orders to that state in the approval slot.


In [ ]:
analysis_destination_figure = plot_destination_states(destination_profiles["inter_state"])
display(analysis_destination_figure)
plt.close(analysis_destination_figure)


RJ, MG, SP, BA and SC account for 962 of the 1,583 late inter-state orders (60.77%). Within these five states, late rates range from 3.73% in SP to 13.09% in RJ. The full state table also shows CE with 68 late deliveries out of 412 orders (16.50%). Among inter-state orders, RJ had more late deliveries, while CE had a higher late rate (16.50% versus 13.09%).


<a id="analysis-state-pairs" name="analysis-state-pairs"></a>

## 11. Seller and customer state pairs

Compare seller → customer states within inter-state orders approved 12:00–18:00. An order enters a state pair only when every item has a known seller state, all seller states agree, and its customer state is known. Orders with multiple seller states or incomplete location information remain in separate coverage groups. State pairs describe endpoints rather than measured carrier routes.


### Coverage of inter-state orders

Count each order once across the three mutually exclusive seller-state groups. Empty groups remain visible with an undefined late rate.


In [ ]:
pair_profiles = summarize_state_pairs(cleaned_items, focus_orders)
display(format_analysis_table(pair_profiles["coverage"]))


Among 19,430 inter-state orders, 19,354 have one complete seller state and account for all 1,583 late deliveries. The remaining 76 orders have sellers in multiple states; none are late. No order in this cohort requires location review.


### Late delivery by seller → customer state pair

Late rate divides late orders by all orders in the pair. Share of inter-state late orders divides by all late inter-state orders in the slot, including orders outside the eligible pairs.


In [ ]:
display(format_analysis_table(pair_profiles["pairs"]))

leading_state_pair = None
if not pair_profiles["pairs"].empty and pair_profiles["pairs"].iloc[0]["late_orders"] > 0:
    leader = pair_profiles["pairs"].iloc[0]
    leading_state_pair = (leader["seller_state"], leader["customer_state"])


In [ ]:
focus_comparisons = summarize_focus_comparisons(pair_profiles["orders"], leading_state_pair)
if leading_state_pair is not None:
    display(format_analysis_table(focus_comparisons["same_destination"]))


SP → RJ has the most late deliveries, with 392 of 2,683 orders (14.61%). It accounts for 24.76% of late inter-state orders in this approval slot and 392 of the 489 late inter-state orders to RJ (80.16%). Other inter-state orders to RJ include 97 late deliveries out of 1,053 (9.21%). Holding the destination state fixed identifies the SP-origin group as a more specific concentration for review.


### Leading state pairs by late order count

The panels compare the same five pairs by late count and within-pair late rate. N gives the total orders in each pair. The dashed line marks the late rate of all inter-state orders in the slot.


In [ ]:
inter_orders = int(pair_profiles["coverage"]["orders"].sum())
inter_late = int(pair_profiles["coverage"]["late_orders"].sum())
inter_late_rate = 100 * inter_late / inter_orders if inter_orders else float("nan")
analysis_state_pairs_figure = plot_state_pairs(pair_profiles["pairs"], inter_late_rate)
display(analysis_state_pairs_figure)
plt.close(analysis_state_pairs_figure)


The five leading pairs account for 724 of 1,583 late inter-state orders (45.74%), and all five have sellers in SP. SP → MG ranks second by late count with 121 of 2,284 orders (5.30%), below the overall inter-state late rate of 8.15%. SP → RJ has a similar order volume but 392 of 2,683 orders late (14.61%). Outcomes differ across destinations within orders originating in the same seller state.


<a id="analysis-focus-time" name="analysis-focus-time"></a>

## 12. Time patterns in the focus cohort

Compare all inter-state orders approved 12:00–18:00 with the leading seller → customer state pair from section 11. Purchase month defines the monthly cohorts; recorded approval hour and weekday define the within-slot views. The state pair is part of the inter-state cohort.


### Late delivery by purchase month

Each row counts all orders purchased in a month within one scope. Months without orders retain zero counts and an undefined rate. The figure shows the full rate range, a closer rate view and monthly order volume; exact late counts remain in the table. The concentration table uses all orders and all late orders of the selected pair across the entire observed period as the respective share denominators.


In [ ]:
focus_time = summarize_focus_time(pair_profiles["orders"], leading_state_pair)
if leading_state_pair is None:
    print("No leading state pair is available. Results cover all inter-state orders only.")
display(format_analysis_table(focus_time["monthly"]))


In [ ]:
analysis_focus_month_figure = plot_focus_months(focus_time["monthly"])
display(analysis_focus_month_figure)
plt.close(analysis_focus_month_figure)


In [ ]:
if leading_state_pair is not None:
    print("Selected pair in November–December 2017 and February–March 2018")
    display(format_analysis_table(focus_comparisons["concentration"]))


For SP → RJ, the purchase periods November–December 2017 and February–March 2018 contain 784 of 2,683 orders (29.22%) and 282 of 392 late deliveries (71.94%). The higher late rates are concentrated in these observed periods. September 2016 has one inter-state order, producing a 100% rate in the full-range panel. The first and last purchase months cover partial observation periods.


### Observed purchase periods and remaining inter-state orders

The periods below were identified from the monthly profiles. Remaining inter-state orders exclude the selected pair. Each rate is calculated from total late orders divided by total orders within its scope and purchase period.


In [ ]:
if leading_state_pair is not None:
    display(format_analysis_table(focus_comparisons["periods"]))


SP → RJ has a late rate of 30.95% in November–December 2017 and 40.97% in February–March 2018, compared with 10.13% and 18.05% among remaining inter-state orders in the same periods. In April–July 2018, the rates are closer at 4.97% and 4.54%. The concentration changes over time, with the earlier purchase periods providing specific historical orders for operational review.


### Matched January–July purchase periods

Compare the same seven purchase months in 2017 and 2018. Each rate uses orders in its own scope and year.


In [ ]:
display(format_analysis_table(focus_time["matched_years"]))


For SP → RJ, 20 of 533 orders purchased in January–July 2017 were late (3.75%), compared with 218 of 1,190 (18.32%) in the same months of 2018. The full inter-state cohort also rose from 146 of 3,891 (3.75%) to 932 of 9,080 (10.26%). These within-dataset comparisons give the pooled SP → RJ rate of 14.61% its time context.


### Late delivery by approval hour and weekday

The six approval hours and seven weekdays each partition their scope once. The tables provide late counts, order denominators and rates; the paired panels show counts and rates on the same categories.


In [ ]:
print("Approval hours")
display(format_analysis_table(focus_time["hours"]))
print("Approval weekdays")
display(format_analysis_table(focus_time["weekdays"]))


In [ ]:
analysis_focus_approval_figure = plot_focus_approval_time(focus_time["hours"], focus_time["weekdays"])
display(analysis_focus_approval_figure)
plt.close(analysis_focus_approval_figure)


Within SP → RJ, 15:00–15:59 has the most late deliveries, with 81 of 469 orders (17.27%). At 14:00–14:59, 72 of 422 are late (17.06%), a gap of only 0.21 percentage points. The observed rates of these two hours are close.

Across all inter-state orders, Monday has the largest late count, 288 of 3,280 orders (8.78%), while Friday has the highest rate, 276 of 2,946 (9.37%). Within SP → RJ, Friday leads both measures, with 75 of 416 orders late (18.03%). The hour and weekday tables summarize separate groupings. These results describe the same historical dataset used to select the leading pair.


<a id="analysis-pair-slots" name="analysis-pair-slots"></a>

## 13. Approval slots within the leading state pair

The leading seller → customer state pair from section 11 is compared across all four approval slots. Each order contributes once, and every item must match the selected pair. Purchase month provides the time comparison.


### Four-slot delivery outcomes

Late counts, order denominators and within-slot rates describe different aspects of the selected pair. The 95% Wilson intervals assume independent orders; shared sellers and purchase dates may weaken that assumption. The pair was selected from the 12:00–18:00 cohort, so these comparisons describe that historical selection.


In [ ]:
pair_all_slots = build_state_pair_orders(cleaned_items, orders, order_routes, leading_state_pair)
pair_slot_results = summarize_state_pair_slots(pair_all_slots)
if leading_state_pair is None:
    print("No leading state pair is available for the four-slot comparison.")
else:
    print(f"Selected state pair: {leading_state_pair[0]} → {leading_state_pair[1]}")
    display(format_analysis_table(pair_slot_results["raw"]))


In [ ]:
if leading_state_pair is not None:
    analysis_pair_slots_figure = plot_state_pair_slots(pair_slot_results["raw"], leading_state_pair)
    display(analysis_pair_slots_figure)
    plt.close(analysis_pair_slots_figure)


For SP → RJ, 12:00–18:00 contains the most late orders (392/2,683), while 00:00–06:00 has the highest within-slot late rate (243/1,535; 15.83%). The afternoon rate is 14.61%. Their descriptive Wilson intervals overlap (14.09–17.74% and 13.32–16.00%, respectively). The larger afternoon count reflects its larger order volume; within this pair, it does not have the highest observed rate.


### Purchase-month comparison

Each purchase month × approval-slot cell shows its order count and late rate. Empty cells have no defined rate. Small denominators limit the interpretation of isolated monthly peaks.


In [ ]:
if leading_state_pair is not None:
    display(format_analysis_table(pair_slot_results["monthly"]))


In [ ]:
if leading_state_pair is not None:
    analysis_pair_months_figure = plot_state_pair_months(pair_slot_results["monthly"], leading_state_pair)
    display(analysis_pair_months_figure)
    plt.close(analysis_pair_months_figure)


SP → RJ has 21 purchase months represented in all four slots. The slot with the highest rate varies by month: 06:00–12:00 leads in November 2017 and March 2018, although it has the lowest pooled rate across the full pair. Elevated rates in November–December 2017 and February–March 2018 appear in several slots rather than only the afternoon. In November 2017, the pair has 707 orders and 226 late deliveries (31.97%); in February 2018, it has 567 orders and 222 late deliveries (39.15%). The smaller observed cohort has the higher late rate, while order count alone does not measure operational workload. October 2016 has just 2–7 orders per slot, so its monthly rates provide little basis for ranking slots.


### Comparison using common purchase-month weights

Monthly standardization includes months represented in every approval slot and gives each slot the same pooled purchase-month weights. The table separates the full-cohort raw rate from the standardized rate and reports each standardized difference from 12:00–18:00 in percentage points. The interior-month check excludes the first and last observed months.


In [ ]:
if leading_state_pair is not None:
    display(format_analysis_table(pair_slot_results["standardized"]))
    if pair_slot_results["standardized"]["standardized_late_rate_pct"].isna().all():
        print("No purchase month contains all four slots; month-standardized rates are undefined.")


All 8,034 SP → RJ orders fall in the 21 shared months. With common purchase-month weights, 00:00–06:00 remains highest at 15.72%, followed by 12:00–18:00 at 14.52%, a 1.20 percentage-point difference. Excluding the first and last observed months leaves 19 common months and the same ordering (16.19% versus 15.17%). The different purchase-month composition of the slots therefore does not reverse their observed ranking. This adjustment covers purchase month only; it does not account for seller, product or other differences within months, nor establish an effect of approval time.


<a id="analysis-route-stages" name="analysis-route-stages"></a>

## 14. Delivery stages within matched destination periods

SP → RJ is compared with other inter-state orders to RJ within three purchase periods and four approval slots. Each order enters one group once, and all items must agree on its destination state.


### Destination coverage and delivery outcomes

Coverage counts all inter-state orders in each purchase period, then separates the selected pair from other orders to the same destination. Outcome rates use orders within each period, group and approval slot.


In [ ]:
if leading_state_pair is None:
    print("No leading state pair is available for the route-stage comparison.")
else:
    route_stage_comparison = compare_route_delivery_stages(
        cleaned_items, orders, order_routes, pair_all_slots, stage_profiles, leading_state_pair,
    )
    display(format_analysis_table(route_stage_comparison["coverage"]))
    display(format_analysis_table(route_stage_comparison["outcomes"]))


For afternoon approvals, SP → RJ has 121/391 late orders (30.95%) in November–December 2017 and 161/393 (40.97%) in February–March 2018. Other inter-state orders to RJ have 24/126 (19.05%) and 35/155 (22.58%) late in those same periods. The comparison keeps the destination and purchase period fixed; it differs from section 12's comparison with all remaining inter-state destinations. In April–July 2018, the corresponding rates are 4.97% and 3.72%.


### Approval to carrier handoff

The table reports all orders, valid intervals, unavailable intervals, median and 75th-percentile hours. Calendar hours measure elapsed time. Eligible-day hours use the calendar convention defined in section 7. The figure plots calendar-hour medians; N labels count all orders, while valid interval counts remain in the table.


In [ ]:
if leading_state_pair is not None:
    display(format_analysis_table(
        route_stage_comparison["stages"].loc[route_stage_comparison["stages"]["stage"].eq("Processing")]
    ))


In [ ]:
if leading_state_pair is not None:
    analysis_route_processing_figure = plot_route_period_medians(
        route_stage_comparison["stages"].loc[route_stage_comparison["stages"]["stage"].eq("Processing")],
        "median_calendar_hours", "Approval to carrier handoff by purchase period and approval slot",
        "Median elapsed hours",
    )
    display(analysis_route_processing_figure)
    plt.close(analysis_route_processing_figure)


Across all four approval slots in November–December 2017, SP → RJ has a longer median approval-to-handoff interval than other inter-state orders to RJ. The group medians move closer in February–March 2018. For afternoon approvals, SP → RJ falls from 67.44 to 54.85 hours between those periods, even as its late rate rises from 30.95% to 40.97%. A longer pre-handoff interval alone therefore does not describe that rise. By April–July 2018, the afternoon medians are 40.66 and 33.93 hours for the pair and destination-matched group, respectively.


### Carrier handoff to customer delivery

The same period, destination and approval-slot groups describe recorded time after handoff. The table includes valid interval counts, median and 75th-percentile elapsed and eligible-day hours. The figure shows elapsed-hour medians.


In [ ]:
if leading_state_pair is not None:
    display(format_analysis_table(
        route_stage_comparison["stages"].loc[route_stage_comparison["stages"]["stage"].eq("Carrier transit")]
    ))


In [ ]:
if leading_state_pair is not None:
    analysis_route_transit_figure = plot_route_period_medians(
        route_stage_comparison["stages"].loc[route_stage_comparison["stages"]["stage"].eq("Carrier transit")],
        "median_calendar_hours", "Carrier handoff to delivery by purchase period and approval slot",
        "Median elapsed hours",
    )
    display(analysis_route_transit_figure)
    plt.close(analysis_route_transit_figure)


SP → RJ has a longer median handoff-to-delivery interval than other inter-state orders to RJ in all four approval slots of February–March 2018. For afternoon approvals, the medians are 477.88 versus 335.43 elapsed hours; the 75th percentiles are 715.94 versus 626.68 hours. Median eligible-day hours are 336.30 versus 239.43, so the difference remains under the project's excluded-day convention. These are not staffed working hours. In April–July 2018, the afternoon elapsed medians converge to 180.18 and 176.62 hours, with a similar convergence across the other slots. The recorded intervals identify where time differs, not why a promise was missed.


### Approval to promised delivery date

Promised days use calendar dates, as in section 7. The table shows the median, 75th percentile, negative windows and unavailable values. The figure plots median days for the same period, group and approval-slot cells.


In [ ]:
if leading_state_pair is not None:
    display(format_analysis_table(route_stage_comparison["promises"]))


In [ ]:
if leading_state_pair is not None:
    analysis_route_promise_figure = plot_route_period_medians(
        route_stage_comparison["promises"], "median_days",
        "Approval to promised date by purchase period and approval slot", "Median calendar days",
    )
    display(analysis_route_promise_figure)
    plt.close(analysis_route_promise_figure)


For afternoon approvals, both groups have a 25-day median promised window in November–December 2017. In February–March 2018, SP → RJ has 24 days versus 29 days for other inter-state orders to RJ, alongside a longer recorded transit median. From February–March to April–July 2018, the SP → RJ transit median falls from 477.88 to 180.18 hours, its promised-window median rises from 24 to 32 days, and its late rate falls from 40.97% to 4.97%. Both changes accompany the lower late rate; these group medians cannot apportion its reduction between them. The records do not show when the promised date was set or whether it changed.


### Delivery stages by observed outcome

Within 12:00–18:00, valid stage intervals are compared between orders later classified as on time or late. This retrospective split locates differences in recorded duration; it does not identify a prospective cause of late status.


In [ ]:
if leading_state_pair is not None:
    afternoon_stage_outcomes = route_stage_comparison["by_outcome"].loc[
        route_stage_comparison["by_outcome"]["approval_slot"].eq("12:00–18:00")
    ]
    display(format_analysis_table(afternoon_stage_outcomes))


Among SP → RJ afternoon approvals, late orders have a longer median recorded transit interval than on-time orders in all three periods. In February–March 2018, the medians are 764.89 versus 306.74 hours, while the pre-handoff medians are 57.70 versus 54.30 hours. Across all orders in that period, SP → RJ has a longer transit median than the destination-matched group (477.88 versus 335.43 hours); among late orders alone, the destination-matched group has the longer median (796.54 versus 764.89 hours). These are different order populations, and transit duration is not the amount of time past the promised date. In April–July 2018, only 29 of 583 SP → RJ afternoon orders are late, yet their transit median is 728.23 hours versus 175.11 hours for on-time orders. The destination-matched late subgroup contains only nine orders; both late subgroup medians warrant caution.


<a id="analysis-synthesis" name="analysis-synthesis"></a>

## 15. Evidence across slots, routes and purchase periods

Combine the leading inter-state pair and its destination-matched comparison across purchase periods. The joint view relates approval-slot outcomes to recorded delivery stages and promised windows, using the cohorts from sections 13–14.


### Joint view of purchase period and approval slot

The two panels compare the selected state pair with other inter-state orders to the same destination. Both use one color scale. Each cell reports its late rate and order denominator; a gray cell has no orders and no defined rate. Purchase period comes from the order purchase date, while slot comes from approval time.


In [ ]:
if leading_state_pair is not None:
    analysis_route_period_slot_figure = plot_route_period_slot_rates(
        route_stage_comparison["outcomes"]
    )
    display(analysis_route_period_slot_figure)
    plt.close(analysis_route_period_slot_figure)


SP → RJ has a higher observed late rate than other inter-state orders to RJ in every displayed approval slot and purchase period. Both groups rise from November–December 2017 to February–March 2018 and fall by April–July 2018 across all four slots. For afternoon approvals, the pair-minus-comparator difference is 11.90, 18.39 and 1.26 percentage points across those periods, respectively. SP → RJ's four slot rates range from 28.49% to 33.82% in the first period, 34.68% to 45.81% in the second, and 4.30% to 4.97% in the third. The changes are therefore not confined to afternoon approvals or to the selected pair. Each cell has its own denominator; the shared pattern does not identify a common operational cause.


### Afternoon cohort and delivery-stage evidence

For 12:00–18:00 approvals, the compact table brings together late outcomes, recorded stage medians and the median window from approval to the promised date. Valid interval counts and 75th percentiles appear in section 14. Stage medians do not sum to the median end-to-end delivery time.


In [ ]:
if leading_state_pair is not None:
    afternoon_route_evidence = summarize_afternoon_route_evidence(route_stage_comparison)
    display(format_analysis_table(afternoon_route_evidence))


In February–March 2018, SP → RJ afternoon orders have a 40.97% late rate versus 22.58% for other inter-state orders to RJ. Their median carrier-to-customer times are 477.88 and 335.43 hours, while median promised windows are 24 and 29 days. Median approval-to-handoff times are closer at 54.85 and 51.73 hours. In April–July 2018, the late rates fall to 4.97% and 3.72%, and transit medians converge to 180.18 and 176.62 hours. The SP → RJ promised-window median also rises to 32 days. Shorter recorded transit and a longer promised window accompany its lower late rate, but their separate contributions cannot be established from group medians. Matching on destination and broad purchase period does not remove seller or within-period month differences; the route and periods were selected after inspecting these data.


<a id="analysis-overnight" name="analysis-overnight"></a>

## 16. Orders approved from 00:00–06:00

Select orders approved from midnight inclusive to 06:00 exclusive. Retain all route types and review unresolved customer destinations separately.


In [ ]:
night_focus_orders, night_focus_diagnostics = build_focus_cohort(
    cleaned_items, orders, order_routes, approval_slot="00:00–06:00",
)
display(format_analysis_table(night_focus_diagnostics["summary"]))

if night_focus_diagnostics["location_review"].empty:
    print("No customer states require review in this cohort.")
else:
    print("Customer states requiring review")
    display(format_analysis_table(night_focus_diagnostics["location_review"]))


The overnight cohort contains 19,910 orders, including 1,455 late deliveries (7.31%). Every order has a consistent customer state across its retained items; no customer destinations require review.


### Route composition

Count each order once across intra-state, inter-state, mixed and unknown routes. Order shares and late-order shares use the complete overnight cohort. Empty route groups retain undefined late rates.


In [ ]:
night_route_summary = summarize_focus_routes(night_focus_orders)
display(format_analysis_table(night_route_summary))


In [ ]:
analysis_overnight_route_figure = plot_focus_routes(night_route_summary, "00:00–06:00")
display(analysis_overnight_route_figure)
plt.close(analysis_overnight_route_figure)


Inter-state deliveries account for 63.53% of overnight orders but 72.99% of late deliveries (1,062 of 1,455). Their late rate is 8.40%, compared with 5.46% for intra-state deliveries. Intra-state orders still contribute 393 late deliveries, or 27.01% of the overnight total. All 63 mixed-route orders arrived on time. The unknown-route group contains no orders, so its late rate is undefined.


### Customer states across all routes

Rank all customer states by late order count, then total orders and state code. Each rate uses orders to that state; both share columns use the complete overnight cohort. Unresolved customer states remain separate.


In [ ]:
night_destination_profiles = summarize_destination_states(night_focus_orders)
display(format_analysis_table(night_destination_profiles["all_routes"]))


SP receives the most late overnight deliveries: 452 of 8,365 orders (5.40%). It accounts for 42.01% of overnight orders and 31.07% of late deliveries. RJ has fewer orders but a higher late rate: 309 of 2,283 (13.53%), representing 11.47% of orders and 21.24% of late deliveries. MG receives slightly more orders than RJ (2,306) but has 120 late deliveries (5.20%). The count ranking and the destination-specific late rates therefore describe different concentrations.


### Inter-state destinations

Restrict the destination table to overnight inter-state orders. Both share columns use the inter-state subset: all its orders and all its late deliveries, respectively.


In [ ]:
display(format_analysis_table(night_destination_profiles["inter_state"]))


RJ leads the inter-state late count with 300 of 2,129 orders (14.09%). It represents 16.83% of inter-state orders and 28.25% of inter-state late deliveries. Only 87 of SP's 452 late deliveries are inter-state; the remaining 365 are intra-state. For RJ, 300 of 309 late deliveries are inter-state. The leading destination consequently depends on whether the scope includes all routes or inter-state routes alone.


### Seller-state coverage

Retain the complete overnight cohort, including mixed and unknown routes, when assessing pair coverage. Assign a pair only when the customer state and a single seller state are complete and consistent. A single seller state may contain multiple sellers. Multiple-state and unresolved orders remain in the coverage totals.


In [ ]:
night_pair_profiles = summarize_state_pairs(
    cleaned_items, night_focus_orders, include_intra_state=True,
)
display(format_analysis_table(night_pair_profiles["coverage"]))


A single seller state is identifiable for 19,792 orders (99.41%), covering 1,454 of the 1,455 late deliveries (99.93%). Another 118 orders have sellers in multiple states, including 63 mixed-route orders and 55 inter-state orders; one of these 118 orders arrived late. They remain in the overnight share denominators and are excluded from individual state-pair rows. No orders require location review.


### Seller → customer state pairs

The complete table includes eligible intra-state and inter-state pairs, ranked by late order count. Rates use orders within each pair. Order shares and late-order shares use all overnight orders and all overnight late deliveries, including orders outside eligible pairs.


In [ ]:
display(format_analysis_table(night_pair_profiles["pairs"]))


Across the 315 eligible pairs, SP → SP has the most late deliveries: 365 of 6,417 orders (5.69%), accounting for 25.09% of all overnight late deliveries. SP → RJ follows with 243 of 1,535 (15.83%), or 16.70% of overnight late deliveries. SP → MG has almost the same order count as SP → RJ (1,534), but 89 late deliveries (5.80%). The SP → SP row corresponds to the intra-state orders destined for SP in the destination tables; those findings refer to the same orders.


<a id="analysis-two-slot-comparison" name="analysis-two-slot-comparison"></a>

## 17. Overnight and afternoon approval cohorts

Compare 00:00–06:00 with 12:00–18:00 within the same customer state or eligible seller → customer state pair. Each cohort retains all route types. Differences are overnight minus afternoon rates, in percentage points.

### Scope and group selection

The displayed groups are the union of:

- The five largest late-order counts in each slot.
- The three highest late rates in each slot among groups with at least 100 orders and a late delivery.
- The largest absolute rate difference among groups with at least 100 orders in both slots.
- SP → RJ as the previously examined reference pair, when present.

Count ties use total orders, then state codes. Rate ties use late counts, total orders, then state codes. Absolute-gap ties use pooled order counts, then state codes. Ranking uses unrounded values. Selected rows are displayed by combined late counts, combined order counts and state codes; remainder and review groups follow them in the figures. The 100-order cutoff is a screening rule, not a guarantee of precision.

Wilson 95% intervals describe binomial precision assuming independent orders. Shared sellers or dates may weaken that assumption. Selection is exploratory; the intervals do not establish pairwise significance. Comparisons do not adjust for purchase month or seller composition.


In [ ]:
afternoon_pair_profiles = summarize_state_pairs(
    cleaned_items, focus_orders, include_intra_state=True,
)
destination_slot_comparison = compare_approval_slot_groups(
    night_focus_orders, focus_orders, group_by="customer_state",
)
state_pair_slot_comparison = compare_approval_slot_groups(
    night_pair_profiles["orders"], afternoon_pair_profiles["orders"], group_by="state_pair",
)

slot_comparison_columns = [
    "group", "approval_slot", "orders", "late_orders", "late_rate_pct",
    "wilson_lower_pct", "wilson_upper_pct", "difference_pp",
]
display(format_analysis_table(destination_slot_comparison["scope_summary"][[
    "approval_slot", "orders", "late_orders", "late_rate_pct",
    "wilson_lower_pct", "wilson_upper_pct", "difference_pp",
]]))


The overnight cohort has 1,455 late deliveries among 19,910 orders (7.31%), versus 2,085 among 30,448 afternoon orders (6.85%). Its late rate is 0.46 percentage points higher, while the afternoon cohort has 630 more late deliveries. Counts and rates identify different priorities.


### Customer-state comparison

Both slots use the same selected destinations. The selection table records why each state is included. Complete results, including unselected states and unresolved destinations, remain in `destination_slot_comparison["all_groups"]`.


In [ ]:
display(format_analysis_table(destination_slot_comparison["selection"]))
display(format_analysis_table(destination_slot_comparison["selected"][slot_comparison_columns]))


In [ ]:
analysis_destination_slot_figure = plot_slot_group_comparison(
    destination_slot_comparison["plot_data"], "customer_state",
)
display(analysis_destination_slot_figure)
plt.close(analysis_destination_slot_figure)


SP and RJ lead the late-order counts in both slots. Their overnight rates exceed their afternoon rates by 0.94 and 1.17 percentage points, respectively. RS has fewer overnight orders than afternoon orders (1,329 versus 1,507), yet more late deliveries (100 versus 80); its overnight rate is 7.52% versus 5.31%. BA is nearly unchanged at 11.20% versus 11.25%.

The higher overall overnight rate does not hold for every destination. CE is 10.38% overnight versus 16.43% in the afternoon, a −6.04-point difference and the largest absolute gap among states with at least 100 orders in both slots. MA has the highest overnight rate among states qualifying for rate ranking (32/181, 17.68%), while its afternoon rate is 18.91%. AL enters through the afternoon ranking: 33/133 deliveries are late (24.81%); its overnight sample has only 71 orders, so it does not qualify for overnight rate ranking or the largest-gap rule.

The other destinations together have rates of 7.05% overnight and 7.31% in the afternoon. Smaller selected groups have wider intervals than the large SP and RJ samples. Neither the observed gaps nor interval overlap alone determines which slot pairs differ statistically.


### Seller → customer state-pair comparison

Include eligible intra-state and inter-state pairs from both slots. Unselected eligible pairs are pooled from their counts in the figure; multiple seller-state and unresolved orders remain separate. The complete table remains in `state_pair_slot_comparison["all_groups"]`.


In [ ]:
display(format_analysis_table(state_pair_slot_comparison["selection"]))
display(format_analysis_table(state_pair_slot_comparison["selected"][slot_comparison_columns]))


In [ ]:
analysis_state_pair_slot_figure = plot_slot_group_comparison(
    state_pair_slot_comparison["plot_data"], "state_pair",
)
display(analysis_state_pair_slot_figure)
plt.close(analysis_state_pair_slot_figure)


SP → SP has the most late deliveries in both slots: 365/6,417 overnight (5.69%) and 460/9,718 in the afternoon (4.73%). SP → RJ has a higher overnight rate, 15.83% versus 14.61% (+1.22 points), but more afternoon late deliveries, 392 versus 243. SP → RS also has more overnight late deliveries (70 versus 57) and a higher rate (7.71% versus 5.89%).

The largest qualifying absolute gap is PR → RS: 11/133 overnight orders are late (8.27%), versus 4/175 afternoon orders (2.29%), a +5.98-point difference. SP → MS has a similar direction, 17/113 (15.04%) versus 14/152 (9.21%), or +5.83 points. Their modest late counts limit how much can be inferred from the observed gaps.

The direction reverses for SP → CE (11.94% overnight versus 16.30% afternoon) and SP → MA (18.58% versus 21.99%). SP → PI is included through its afternoon rate ranking; its overnight rate of 18.75% uses only 64 orders and does not qualify for the overnight ranking. Other eligible pairs together have much closer rates, 6.49% versus 6.36%. Multiple seller-state orders contribute one late delivery in each slot.

The selected pairs account for 945 of the 1,455 overnight late deliveries (64.95%). The pooled eligible pairs account for another 509 (34.98%), and orders with sellers in multiple states account for the remaining one. Across the full observation period, SP → SP and SP → RJ together account for 608 overnight late deliveries (41.79%). [Section 19](#analysis-slot-stage-context) examines all overnight SP → SP orders and limits the SP → RJ stage comparison to three purchase periods. The 42 late SP → RJ orders outside those periods remain in the full-route and monthly comparisons.


<a id="analysis-slot-months" name="analysis-slot-months"></a>

## 18. Purchase-month comparisons across approval slots

Inspect the full purchase-month history of the leading groups from the two-slot comparison. Approval time defines the cohort; purchase time defines the month.

Select one late-count leader and one qualifying late-rate leader per slot, the largest qualifying absolute gap, and SP → RJ when present. The ranking and 100-order screening rules follow section 17. The union contains at most five customer states or six state pairs. Every selected group retains the complete observed month range, including zero-order cells.


### Customer-state monthly patterns

Keep all selected states in the monthly table. The figure focuses on CE and AL to show the destination gap and sparse monthly samples. SP and MA remain available in the complete table; the pair figure examines SP → SP and SP → MA separately.


In [ ]:
destination_month_results = summarize_slot_group_months(
    night_focus_orders, focus_orders, group_by="customer_state",
)
display(format_analysis_table(destination_month_results["selection"]))
display(format_analysis_table(destination_month_results["monthly"]))


In [ ]:
destination_month_plot_groups = [
    group for group in ("CE", "AL")
    if group in destination_month_results["selection"]["group"].tolist()
]
analysis_destination_month_figure = plot_slot_group_months(
    destination_month_results["monthly"], group_by="customer_state",
    groups=destination_month_plot_groups or None,
)
display(analysis_destination_month_figure)
plt.close(analysis_destination_month_figure)


CE's afternoon late rate reaches 40.00% in November 2017 (12/30) and 53.85% in March 2018 (21/39), versus 12.50% (2/16) and 33.33% (4/12) overnight. The direction is not uniform: in January 2018, overnight is 25.00% (2/8) versus 10.00% (3/30). Across the 20 common months, overnight is higher in five, afternoon in six, and both rates are equal in nine.

The early overnight peaks in March 2017 are based on 2/3 orders for CE (66.67%) and 1/1 for AL (100.00%). AL's 80.00% afternoon rate in July 2017 represents only four late deliveries among five orders. In February 2018 it has five late afternoon deliveries among 14 orders, with no overnight orders available for that month. The count panel distinguishes small denominators from sustained volume. MA also has concentrated peaks: November 2017 records 10/18 late overnight deliveries (55.56%) and 8/14 in the afternoon (57.14%). A high monthly rate in these groups should be read with its order count.

SP has a higher overnight rate in 17 of its 21 common months. For August 2018 purchases, its rates are 11.69% overnight (74/633) and 10.04% afternoon (107/1,066), based on much larger samples than the smaller-state monthly peaks.


### State-pair monthly patterns

The table covers every selected pair. Plot SP → SP for the largest late counts, SP → MA for the qualifying rate leaders, and PR → RS for the largest qualifying gap. SP → RJ retains its full monthly table here; its four-slot monthly figure is already available in section 13. Plots use the available preferred groups, or all selected groups when none of those preferences are present.


In [ ]:
state_pair_month_results = summarize_slot_group_months(
    night_pair_profiles["orders"], afternoon_pair_profiles["orders"], group_by="state_pair",
)
display(format_analysis_table(state_pair_month_results["selection"]))
display(format_analysis_table(state_pair_month_results["monthly"]))


In [ ]:
state_pair_month_plot_groups = [
    group for group in ("SP → SP", "SP → MA", "PR → RS")
    if group in state_pair_month_results["selection"]["group"].tolist()
]
analysis_state_pair_month_figure = plot_slot_group_months(
    state_pair_month_results["monthly"], group_by="state_pair",
    groups=state_pair_month_plot_groups or None,
)
display(analysis_state_pair_month_figure)
plt.close(analysis_state_pair_month_figure)


SP → SP has a higher overnight rate in 18 of its 21 common months. August 2018 has its largest monthly late count in both slots: 69/520 overnight orders are late (13.27%), versus 98/840 afternoon orders (11.67%). These 69 and 98 cases represent 18.90% and 21.30% of the route's overnight and afternoon late totals, respectively. The afternoon count is larger even in a month with a higher overnight rate. August is the final observed purchase month; the peak alone does not establish a continuing trend. The isolated 12.50% overnight point in October 2016 represents only 1/8 orders.

SP → RJ has a higher overnight rate in 12 common months, a higher afternoon rate in seven, and equal rates in two. Its February 2018 rates are 49.33% overnight (37/75) and 42.29% in the afternoon (85/201). The SP → SP count peak in August differs from SP → RJ's concentration in November–December 2017 and February–March 2018. Historical review periods therefore need to remain specific to each route.

SP → MA has fewer than ten orders in 35 of its 42 observed month-slot cells. Its 100.00% afternoon rate in January 2017 is a single late delivery among one order. November 2017 has seven late overnight deliveries among 12 orders (58.33%) and eight among 12 afternoon orders (66.67%); February 2018 reverses that direction, at 5/9 (55.56%) versus 1/9 (11.11%). The whole-period ranking does not hold in every month.

For PR → RS, overnight is higher in seven common months, afternoon in two, and both rates are equal in 11. Four of its 11 overnight late deliveries are March 2018 purchases (4/14, 28.57%); the afternoon group has 0/8 in that month. The gap is visible beyond one month, but it is based on only 11 overnight and four afternoon late deliveries over the entire period.


### Common months and customer-state rates

For each group, retain months containing at least one order in both slots. A month's weight equals its pooled order count across the two slots divided by the pooled total across those common months. Apply the same weights to each slot's monthly late rates.

Read the full-cohort raw rate, common-month raw rate and standardized rate separately. Coverage reports both orders and late deliveries included or excluded. Weights are group-specific and remain in each result's `weights` table. No common months give undefined standardized rates; one common month gives a comparison limited to that month. Weighting does not remove uncertainty from sparse cells or control seller and within-month differences.


In [ ]:
month_rate_columns = [
    "group", "approval_slot", "common_months", "raw_late_rate_pct",
    "common_raw_late_rate_pct", "standardized_late_rate_pct",
]
month_coverage_columns = [
    "group", "approval_slot", "included_orders", "excluded_orders",
    "included_late_orders", "excluded_late_orders",
]

display(format_analysis_table(destination_month_results["standardized"][month_coverage_columns]))
display(format_analysis_table(destination_month_results["standardized"][month_rate_columns]))
display(format_analysis_table(destination_month_results["differences"]))


For CE, the overnight-minus-afternoon gap is −6.04 percentage points over the full cohort, −5.96 within the 20 common months, and −2.84 after applying common weights. Only four afternoon orders, including one late delivery, are outside those common months. Most of the numerical change therefore occurs when the retained monthly rates receive common weights.

AL has 18 common months. Its afternoon comparison excludes 14 orders and five late deliveries, all from February 2018, while the overnight comparison excludes one on-time order. The gap changes from −6.50 points over the full cohort to −4.96 within common months, then −4.27 after weighting. Its standardized rates are 18.19% overnight and 22.46% afternoon.

SP's standardized gap remains positive at +1.01 points across 21 common months. MA's gap is negative at −2.38 points, compared with −1.23 before weighting; all its orders are retained. Month standardization changes the size of these gaps, but the four selected states retain their respective whole-period directions.


### State-pair rates under common month weights

Use the same calculation for each pair. Two-slot weights pool only overnight and afternoon orders. The four-slot comparison in section 13 pools all four slots, so its standardized values need not match this table.


In [ ]:
display(format_analysis_table(state_pair_month_results["standardized"][month_coverage_columns]))
display(format_analysis_table(state_pair_month_results["standardized"][month_rate_columns]))
display(format_analysis_table(state_pair_month_results["differences"]))


No orders or late deliveries are excluded from the common-month comparison for any of the four selected pairs. SP → SP and SP → RJ each have 21 common months. Their standardized overnight/afternoon rates are 5.71%/4.71% and 15.89%/14.62%, respectively: gaps of +1.00 and +1.27 percentage points.

SP → MA also has 21 common months. Its standardized rates are 18.54% overnight and 23.18% afternoon, widening the gap from −3.40 to −4.64 points. PR → RS has 20 common months; its gap changes from +5.98 to +5.58 points, with standardized rates of 7.59% and 2.01%. None of the selected pair gaps reverses after common weighting. This adjustment addresses purchase-month composition within each pair; the small late counts on PR → RS and sparse monthly samples on SP → MA still limit interpretation.


<a id="analysis-slot-stage-context" name="analysis-slot-stage-context"></a>

## 19. Delivery stages and promises for overnight and afternoon approvals

Read SP → RJ and other inter-state deliveries to RJ within the same purchase periods used in section 14. The comparison separates approval time from purchase period. SP → SP is then examined within its own route because its overnight excess appears in 18 of 21 common purchase months.


### SP → RJ within destination-matched purchase periods

The compact table uses the existing route-stage results. Both approval slots share the destination definition and period boundaries. The three periods cover 823 of 1,535 overnight SP → RJ orders, including 201 of its 243 late deliveries; for afternoon approvals they cover 1,367 of 2,683 orders and 311 of 392 late deliveries. Orders outside these periods remain in the full-route and monthly comparisons.


In [ ]:
night_afternoon_route_evidence = None
if leading_state_pair is None:
    print("No selected inter-state pair is available for the destination-matched comparison.")
else:
    night_afternoon_route_evidence = summarize_route_slot_evidence(
        route_stage_comparison, approval_slots=COMPARISON_SLOTS,
    )
    display(format_analysis_table(night_afternoon_route_evidence))


For SP → RJ, overnight late rates are 33.82%, 45.81% and 4.60% across the three purchase periods, compared with 30.95%, 40.97% and 4.97% in the afternoon. The overnight excess is therefore concentrated in the first two periods rather than present in every period. Other inter-state orders to RJ also rise and fall, at 15.91%, 32.88% and 2.38% overnight.

From November–December 2017 to February–March 2018, the overnight SP → RJ processing median falls from 66.77 to 45.41 hours while the carrier-transit median rises from 375.68 to 526.61 hours. Its late rate increases despite the shorter interval before handoff. In February–March, the overnight transit median exceeds the afternoon value of 477.88 hours, although both slots have a 24-day median promised window. In April–July, the transit medians become similar (183.83 versus 180.18 hours), and overnight no longer has the higher late rate.


### Stage distributions and promised windows

Keep valid and unavailable interval counts beside medians and P75 values. Calendar hours measure elapsed time; eligible-day hours follow the calendar convention in section 7. Figures in [section 14](#analysis-route-stages) and [section 15](#analysis-synthesis) already display these periods across all four approval slots, so the tables below isolate the two-slot comparison without repeating those figures.


In [ ]:
if night_afternoon_route_evidence is not None:
    night_afternoon_route_stages = route_stage_comparison["stages"].loc[
        route_stage_comparison["stages"]["approval_slot"].isin(COMPARISON_SLOTS)
    ].copy()
    night_afternoon_route_promises = route_stage_comparison["promises"].loc[
        route_stage_comparison["promises"]["approval_slot"].isin(COMPARISON_SLOTS)
    ].copy()
    for stage in ("Processing", "Carrier transit"):
        print(stage)
        display(format_analysis_table(
            night_afternoon_route_stages.loc[night_afternoon_route_stages["stage"].eq(stage)]
        ))
    display(format_analysis_table(night_afternoon_route_promises))


In February–March 2018, SP → RJ overnight transit has a median of 526.61 hours and P75 of 849.51, versus 382.25 and 672.00 hours for other inter-state orders to RJ. The median eligible-day values are 382.61 versus 270.08 hours, so the destination-matched contrast remains under that convention. All displayed stage intervals are valid, and all promised windows are available and nonnegative.

The overnight SP → RJ promised-window median is 23 days in November–December, versus 25 days in the afternoon and 25 days for the overnight comparator. In February–March it is 24 days for both overnight destination groups, while the afternoon comparator has 29 days. Thus the afternoon five-day difference between route groups does not also describe the overnight comparison. By April–July the SP → RJ window is 30 days overnight versus 32 in the afternoon, alongside much shorter transit times in both slots. These are joint descriptive changes; their separate contributions to late delivery are not identified.


### Overnight stages by observed delivery outcome

Separate on-time and late overnight deliveries within the same destination groups and purchase periods. The afternoon split remains in section 14. These retrospective groups describe recorded durations after the delivery outcome is known; transit time is not the number of hours past the promised date.


In [ ]:
if night_afternoon_route_evidence is not None:
    night_route_stage_outcomes = route_stage_comparison["by_outcome"].loc[
        route_stage_comparison["by_outcome"]["approval_slot"].eq(COMPARISON_SLOTS[0])
    ].copy()
    display(format_analysis_table(night_route_stage_outcomes))


For SP → RJ overnight approvals, the February–March transit medians are 891.75 hours among 93 late orders and 308.95 among 110 on-time orders. The late destination-matched comparator has a median of 854.75 hours across 24 orders. This differs from the afternoon late-only comparison, where the comparator's median exceeds SP → RJ's: a ranking based on all orders need not hold within late-only groups or across slots.

In April–July, 16 of 348 overnight SP → RJ orders are late. Their transit median remains 672.54 hours, versus 178.11 among the 332 on-time orders. The overnight comparator has only three late orders, so its duration summary is especially sensitive to individual records. The lower period-wide rate does not mean that long individual delivery intervals have disappeared.


### SP → SP within common purchase months

SP → SP contributes 365 overnight late deliveries, more than SP → RJ's 243, and its overnight rate remains higher after month weighting. This warrants a within-route comparison of recorded stages and promises. Select SP → SP using the existing complete-item rule, retain the two approval slots and their shared purchase months, and summarize delivery outcomes and stage profiles.

The comparison keeps one route throughout; it does not use the inter-state destination comparator. The stage and promise summaries below pool the retained orders without applying month weights or matching sellers. The month-standardized late-rate comparison remains in section 18.


In [ ]:
sp_sp_common_months = state_pair_month_results["weights"].loc[
    state_pair_month_results["weights"]["group"].eq("SP → SP"), "purchase_month"
]
sp_sp_stage_profiles = None
if sp_sp_common_months.empty:
    print("No shared purchase months are available for the SP → SP stage comparison.")
else:
    sp_sp_all_slots = build_state_pair_orders(
        cleaned_items, orders, order_routes, ("SP", "SP"), include_intra_state=True,
    )
    sp_sp_comparison_orders = sp_sp_all_slots.loc[
        sp_sp_all_slots["approval_slot"].isin(COMPARISON_SLOTS)
        & sp_sp_all_slots["order_purchase_timestamp"].dt.strftime("%Y-%m").isin(sp_sp_common_months)
    ].copy()
    sp_sp_stage_profiles = investigate_delivery_stages(sp_sp_comparison_orders)
    sp_sp_slot_summary = summarize_delivery_by_slot(sp_sp_comparison_orders)
    display(format_analysis_table(sp_sp_slot_summary.loc[
        sp_sp_slot_summary["approval_slot"].isin(COMPARISON_SLOTS),
        ["approval_slot", "orders", "late_orders", "late_rate_pct"],
    ]))
    display(format_analysis_table(sp_sp_stage_profiles["quality"]))


All 6,417 overnight and 9,718 afternoon SP → SP orders fall within the 21 common months, with late rates of 5.69% and 4.73%. Both recorded stages are valid for every retained order. This is a separate concentration from SP → RJ. Both remain descriptive groups rather than demonstrated causes of the overall slot difference.


### SP → SP whole-cohort stage distributions

Pool all retained orders within each approval slot before separating delivery outcomes. Counts include every order; medians and P75 use valid intervals only, with linear interpolation for P75. Promised-window medians use the same order cohort. These are unweighted summaries of the common-month records.


In [ ]:
if sp_sp_stage_profiles is not None:
    sp_sp_whole_stage_summary = summarize_stage_slots(
        sp_sp_stage_profiles["records"], approval_slots=COMPARISON_SLOTS,
    )
    display(format_analysis_table(sp_sp_whole_stage_summary))
    display(format_analysis_table(
        sp_sp_stage_profiles["promised_summary"].loc[
            sp_sp_stage_profiles["promised_summary"]["approval_slot"].isin(COMPARISON_SLOTS),
            ["approval_slot", "orders", "median_days"],
        ]
    ))


Across all retained SP → SP orders, overnight approval-to-handoff time has a shorter calendar-hour median than afternoon approvals (39.53 versus 47.73 hours). Under the eligible-day convention, the medians are 32.10 versus 30.89 hours, so the calendar-hour ordering does not persist. This distinction reflects the two time conventions rather than measured staffed hours.

Overnight carrier transit has a median of 89.71 calendar hours versus 83.67 in the afternoon, a 6.04-hour difference. The P75 values are much closer, at 147.06 versus 146.61 hours; eligible-day medians are 65.16 versus 61.83 hours. Both stages have 6,417 valid overnight intervals and 9,718 valid afternoon intervals, with none unavailable. The overnight promised-window median is shorter (16 versus 17 calendar days).

These whole-cohort distributions describe all orders in each slot. They provide context for the higher overnight late rate but do not isolate the contribution of processing, transit or promised dates. The retained months are shared; their order weights and seller composition are not matched.


### SP → SP stages by observed delivery outcome

Split the same orders into deliveries later classified as on time or late. Each row now describes an outcome subgroup, so its median has a different population from the whole-cohort table.


In [ ]:
if sp_sp_stage_profiles is not None:
    sp_sp_stage_columns = [
        "stage", "approval_slot", "is_late", "orders", "evaluated",
        "median_calendar_hours", "median_eligible_hours",
    ]
    display(format_analysis_table(sp_sp_stage_profiles["summary"][sp_sp_stage_columns]))


Among on-time SP → SP deliveries, overnight processing is shorter in calendar hours (38.34 versus 46.99), while the eligible-day medians are nearly identical (30.53 versus 30.41). Among late deliveries, overnight processing is longer (112.06 versus 101.44 calendar hours).

Carrier-transit medians are close within each outcome group: 81.64 versus 79.49 hours for on-time orders and 294.41 versus 300.43 for late orders, with overnight listed first. The higher overnight late rate therefore does not coincide with uniformly longer recorded transit within these outcome groups. Promise windows provide additional context, but neither pooled medians nor a split by eventual outcome can establish their effect on the rate.

The smaller PR → RS comparison contains only 11 overnight and four afternoon late deliveries, while SP → MA has fewer than ten orders in 35 of 42 observed month-slot cells. Their monthly patterns remain visible in section 18; these results do not justify another period-by-stage breakdown here. The SP → SP comparison adds a distinct within-route view, while SP → RJ reuses the existing destination-matched evidence. Other routes remain outside this stage-level examination.


<a id="analysis-answer" name="analysis-answer"></a>

## 20. Answer to the delivery question

Connect the overall slot ranking with the route, purchase-month and delivery-stage evidence. Approval time defines the slots; the observations describe delivered orders retained after cleaning.


### Approval slots: late counts and late rates

**12:00–18:00 has the most late deliveries: 2,085 among 30,448 orders (6.85%).** It contains 32.03% of all 6,509 late deliveries and is the primary focus when prioritizing the number of cases. **00:00–06:00 has the highest late rate: 1,455 among 19,910 orders (7.31%).** Its rate is 0.46 percentage points higher, while the afternoon has 630 more late deliveries. The recorded approval slot identifies an association with delivery performance; it does not establish that the hour itself caused lateness.


### Route and purchase-month concentrations

SP → SP and SP → RJ are the two largest eligible state pairs by late count in both slots. Across the full observation period, these mutually exclusive pairs contain **608 overnight late deliveries (41.79% of overnight cases; 9.34% of all late deliveries)** and 852 afternoon late deliveries (40.86% of afternoon cases). The wider selected-pair comparison contains 945 overnight late deliveries (64.95%). The 847 overnight cases outside the two pairs include 337 from other selected pairs, 509 from the pooled eligible pairs and one from the multiple seller-state group.

The stage comparison in [section 19](#analysis-slot-stage-context) covers 566 of the 1,455 overnight late deliveries (38.90%) on the two target routes. It covers all 365 on SP → SP and, within the three purchase periods, 201 on SP → RJ. Including 41 late deliveries from other inter-state orders to RJ in the same periods brings the stage comparison to 607 overnight late deliveries (41.72%).

Both routes retain higher overnight rates after applying common purchase-month weights to the two slots. The standardized rates are 5.71% overnight versus 4.71% afternoon for SP → SP, and 15.89% versus 14.62% for SP → RJ. Purchase-month weighting preserves these directions, but the direction is not universal across destinations: CE still has a higher afternoon rate, with a standardized overnight-minus-afternoon gap of −2.84 percentage points. PR → RS has a larger positive gap but only 11 overnight and four afternoon late deliveries; its scale and uncertainty differ from the large SP samples.

Within SP → RJ, November–December 2017 and February–March 2018 contain 185 overnight late deliveries and 282 afternoon late deliveries. These represent 76.13% and 71.94% of the respective route-slot late totals, but only 12.71% and 13.53% of all late deliveries in their slots. The historical concentration appears in both slots and does not account for most late deliveries across either whole cohort. SP → SP instead has its largest monthly late counts in August 2018, the final observed purchase month, so the routes have different historical periods to prioritize.


### Recorded stages and promised dates

In February–March 2018, SP → RJ has higher overnight than afternoon lateness (45.81% versus 40.97%) and a longer median carrier-transit interval (526.61 versus 477.88 hours), despite the same 24-day median promised window. Its overnight processing median falls from 66.77 hours in November–December to 45.41 hours in February–March while lateness rises. By April–July, transit medians converge and overnight no longer has the higher rate. The destination-matched inter-state group also rises and falls across these periods, so the variation extends beyond one slot and one route.

For SP → SP, pooling all orders yields a 6.04-hour higher overnight median carrier-transit time, while P75 is nearly unchanged. Its promised-window median is 16 days overnight and 17 days in the afternoon. Approval-to-handoff medians are shorter overnight in calendar hours but slightly longer under the eligible-day convention. Within the retrospective on-time and late subgroups, carrier-transit medians are close across slots; these are different populations from the whole-cohort comparison. Both views pool common-month orders without month weighting or seller matching. They describe delivery characteristics but do not isolate the effect of the promised window or approval time.


### Operational priorities and scope

The afternoon provides the larger pool of late cases to review. SP → RJ supplies a specific historical route-and-period concentration for checking carrier-handoff records, subsequent transit and promised dates. SP → SP adds a substantial intra-state group whose overnight rate remains higher under month weighting and whose promised windows merit comparison within the route.

These priorities identify records for operational review rather than a quantified improvement from changing approval hours. The route and period comparisons are exploratory, sellers and within-month differences remain uncontrolled, and eligible-day hours are a calendar convention rather than staffed working hours. Stage medians cannot be added to obtain a median end-to-end journey. The evidence supports several co-occurring delivery characteristics; it does not establish a single main cause or divide lateness into causal contributions.
